<a href="https://colab.research.google.com/github/dhaev/Data-projects/blob/main/freight_stream_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install curl_cffi

In [2]:
from google.colab import drive
drive.mount('/content/drive')
from curl_cffi import requests
from datetime import date
import json
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import duckdb


## track_active_or_changing_loads_sql

## main code

### Libraries

In [4]:
import json
import asyncio
import random
import logging
from datetime import datetime, date
from pathlib import Path
from typing import Dict, Any, List
import pandas as pd
import aiohttp
import numpy as np
import duckdb

# --- Dependency Management ---
# Use the following libraries:
# aiohttp
# pandas
# duckdb
# pyarrow
# nest_asyncio


### Config

In [5]:
# --- Configuration ---
SECRETS_FILE = '/content/drive/MyDrive/freight_analysis/secrets.json'
# Log files and directories
LOG_DIR = Path('/content/logs')
LOG_DIR.mkdir(exist_ok=True)
BRONZE_DIR = Path("/content/drive/MyDrive/freight_analysis/data/bronze")
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_DIR / 'freight_stream.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# File paths
ZONE_FILE = '/content/drive/MyDrive/freight_analysis/zone.csv'
NEXTLOAD_FACTORS_FILE = '/content/drive/MyDrive/freight_analysis/nextload_broker_info_2025-08-23.json'
TRUCKS_FACTORS_FILE = '/content/drive/MyDrive/freight_analysis/trucksmarter_factoring.json'
DUCKDB_PATH = '/content/freight_data.duckdb'
# DUCKDB_PATH = '/content/drive/MyDrive/freight_analysis/freight_data.duckdb'
table_name = 'freight_data'
# Intervals (in seconds)
INTERVAL_NL = 3600
INTERVAL_TS = 3600
# Other constants
STATES = [
    'CT','MA','ME','NH','NJ','RI','VT','DE','NY','PA','DC',
    'MD','NC','SC','VA','WV','AL','FL','GA','MS','TN','IN',
    'KY','MI','OH','IA','MN','MT','ND','SD','WI','IL','KS',
    'MO','NE','AR','LA','OK','TX','AZ','CO','ID','NM','NV',
    'UT','WY','AK','CA','OR','WA',
]
STANDARDIZE_EQUIPMENT_MAP = {
    'StepDeck': 'Step Deck',
    'GooseNeck': 'Removable Gooseneck',
    'Gooseneck': 'Removable Gooseneck',
    'Van': 'Dry Van',
    'BoxTruck': 'Straight Truck',
    'Straight Box': 'Straight Truck',
    'CargoVan': 'Cargo Van'
}

### Database and Data Loading Functions

In [6]:
# --- Database and Data Loading Functions ---
def _load_duckdb_table(duckdb_path: str, table_name: str) -> None:
    """Initializes the DuckDB table if it doesn't exist."""
    column_defs = """
        source VARCHAR, source_id VARCHAR, load_reference VARCHAR,
        post_date TIMESTAMP WITH TIME ZONE, last_updated TIMESTAMP WITH TIME ZONE,
        pickup_id DOUBLE, pickup_city VARCHAR, pickup_state VARCHAR,
        pickup_country VARCHAR, pickup_latitude DOUBLE, pickup_longitude DOUBLE,
        pickup_date TIMESTAMP WITH TIME ZONE, drop_id DOUBLE, drop_city VARCHAR,
        drop_state VARCHAR, drop_country VARCHAR, drop_latitude DOUBLE,
        drop_longitude DOUBLE, drop_date TIMESTAMP WITH TIME ZONE,
        equipment_type VARCHAR, load_length BIGINT, load_weight BIGINT,
        load_height BIGINT, load_width BIGINT, rate BIGINT, comment VARCHAR,
        contact_name VARCHAR, contact_phone VARCHAR, contact_email VARCHAR,
        contact_fax VARCHAR, company_name VARCHAR, company_email VARCHAR,
        MC BIGINT, DOT BIGINT, estimated_distance BIGINT, hash VARCHAR,
        pickup VARCHAR, drop VARCHAR, drop_zone VARCHAR, pickup_zone VARCHAR,
        rate_per_mile DOUBLE, apex_credit_score VARCHAR,
        trucksmarter_credit_score VARCHAR, city_lane VARCHAR,
        state_lane VARCHAR, zone_lane VARCHAR
    """
    try:
        with duckdb.connect(database=duckdb_path, read_only=False) as conn:
            conn.execute(f"CREATE TABLE IF NOT EXISTS {table_name} ({column_defs});")
        logger.info(f"DuckDB table '{table_name}' is ready.")
    except Exception as e:
        logger.error(f"Failed to create/connect to DuckDB table: {e}")

def _check_if_exists_in_db(db_path: str, table_name: str, source: str, source_id: str | None, load_hash: str | None) -> bool:
    """Queries the database to check for an existing record."""
    try:
        with duckdb.connect(database=db_path, read_only=True) as conn:
            if source == 'nextload' and load_hash:
                result = conn.execute(f"SELECT COUNT(*) FROM {table_name} WHERE hash = ?;", (load_hash,)).fetchone()
            elif source == 'trucksmarter' and source_id:
                result = conn.execute(f"SELECT COUNT(*) FROM {table_name} WHERE source_id = ?;", (source_id,)).fetchone()
            else:
                return False
            return result[0] > 0
    except Exception as e:
        logger.error(f"Error checking database for existing record: {e}")
        return False

### track_load_updates

In [7]:
# track_active_or_changing_loads_sql('/content/freight_data.duckdb', ['160532662'], ['Flatbed','Step Deck','Reefer','Dry Van'])

In [8]:
# _load_duckdb_table('/content/drive/MyDrive/freight_analysis/freight_data.duckdb', 'freight_data')

# with duckdb.connect('/content/freight_data.duckdb', read_only=False) as conn:
#             # results = conn.execute("""SELECT load_reference, equipment_type, rate, from freight_data where load_reference='120963442' 	""").fetchdf()
#             results = conn.execute("""SELECT load_reference, count(load_reference) from freight_data group by load_reference Having count(load_reference) > 2""").fetchdf()
# results

In [9]:
# --- Revised SQL Function with DataFrame Input ---
def track_active_or_changing_loads_sql(db_path: str, record_df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """
    Finds loads with a price change or where half the time to pickup has passed.
    This version now correctly handles combinations of search criteria by
    processing the input DataFrame directly.
    """
    if record_df.empty or not columns:
        return pd.DataFrame()

    # Generate unique combinations from the specified columns
    combinations = record_df[columns].drop_duplicates().values.tolist()

    # Generate the WHERE clause and corresponding values
    where_clauses = []
    values = []
    for combo in combinations:
        clause = " AND ".join([f"{col} = ?" for col in columns])
        where_clauses.append(f"({clause})")
        values.extend(combo)

    where_clause_str = " OR ".join(where_clauses)

    query = f"""
    WITH load_history AS (
        SELECT
            load_reference,
            city_lane,
            zone_lane,
            state_lane,
            company_name,
            "MC",
            equipment_type,
            load_weight,
            post_date,
            pickup_date,
            last_updated,
            rate,
            FIRST_VALUE(rate) OVER (PARTITION BY city_lane, equipment_type, load_weight, pickup_date, company_name ORDER BY last_updated ASC) AS first_rate,
            LAST_VALUE(rate) OVER (PARTITION BY city_lane, equipment_type, load_weight, pickup_date ORDER BY last_updated ASC) AS last_rate,
            COUNT(DISTINCT last_updated) OVER (PARTITION BY city_lane, equipment_type, load_weight, pickup_date, company_name) AS posted_count,
            date_diff('day', post_date, pickup_date) AS total_days_to_pickup,
            date_diff('day', post_date, last_updated) AS days_since_post,
            ROW_NUMBER() OVER (
                PARTITION BY city_lane, equipment_type, load_weight, pickup_date, company_name
                ORDER BY last_updated DESC
            ) AS row_num,
            trucksmarter_credit_score,
            apex_credit_score,
            comment
        FROM freight_data
        WHERE
            {where_clause_str}
    )
    SELECT
        *
    FROM load_history
    WHERE row_num = 1
    ORDER BY last_updated DESC;
    """

    try:
        with duckdb.connect(database=db_path, read_only=False) as conn:
            results = conn.execute(query, tuple(values)).fetchdf()
        return results
    except Exception as e:
        logger.error(f"Error executing analytic query: {e}")
        return pd.DataFrame()

### broker info

In [10]:
def _broker_info(df, factoring_df, source, mc_join_col = None):
    """
    Merge official broker names and credit scores from factoring_df into a load-level DataFrame.
    """
    logger.info(f"Starting broker renaming for source: {source}")
    try:
        # Reset index to access MC and DOT as columns
        factoring_df = factoring_df.reset_index()

        if source == "nextload":
            factoring_df_clean = factoring_df[['MC', 'legalName', 'apex_credit_score', 'trucksmarter_credit_score']].copy()
            join_key = mc_join_col or 'MC'
            factoring_df_clean['MC'] = pd.to_numeric(factoring_df_clean['MC'], errors='coerce').round().astype('Int64')
            df[join_key] = pd.to_numeric(df[join_key], errors='coerce').round().astype('Int64')

            merged = df.merge(factoring_df_clean, how='left', on='MC')
            merged['company_name'] = merged['legalName'].fillna(merged.get('company_name'))
            merged = merged.drop(columns=['legalName'])

        elif source == "trucksmarter":
            factoring_df_clean = factoring_df[['MC','DOT','brokerId','legalName', 'apex_credit_score', 'trucksmarter_credit_score']].rename(columns={'brokerId': 'company_name'}).copy()
            join_key = mc_join_col or 'company_name'
            merged = df.merge(factoring_df_clean, how='left', on='company_name')
            merged['company_name'] = merged['legalName'].fillna(merged.get('company_name'))
            merged = merged.drop(columns=['legalName',])

        else:
            raise ValueError(f"Unknown source '{source}'. Must be 'nextload' or 'trucksmarter'.")

        logger.info("Broker renaming completed successfully.")
        return merged

    except Exception as e:
        logger.error(f"Error during broker renaming: {e}")
        raise

### enrich data

In [11]:


def enrich_record(df: pd.DataFrame, source: str, factoring_df: pd.DataFrame, equipment_map: Dict[str, str], zone_map: pd.Series, country_fix: bool = False) -> pd.DataFrame:
    """Enriches a single load record with derived fields."""
    logger.info(f"Starting enrichment for a new {source} record.")

    # Check if the input is empty or invalid
    if df is None or not df:
        logger.error("Input data is empty or invalid.")
        return pd.DataFrame()

    try:
        record_df = pd.DataFrame([df]).copy()
        record_df = record_df.explode('equipment_type', ignore_index=True).dropna(subset=['equipment_type'])
        record_df['equipment_type'] = record_df['equipment_type'].replace(equipment_map)
        record_df['source_id'] = record_df['source_id'].astype(str)

        if country_fix:
            record_df['pickup_country'] = record_df['pickup_country'].replace({'US': 'USA', 'us': 'USA'})
            record_df['drop_country'] = record_df['drop_country'].replace({'US': 'USA', 'us': 'USA'})

        record_df['pickup'] = record_df[['pickup_city', 'pickup_state', 'pickup_country']].astype(str).agg(','.join, axis=1)
        record_df['drop'] = record_df[['drop_city', 'drop_state', 'drop_country']].astype(str).agg(','.join, axis=1)

        if source == 'trucksmarter':
          try:
              for col in ['pickup_date', 'post_date', 'last_updated', 'drop_date']:
                  if col in record_df.columns:
                      dt_series = pd.to_datetime(record_df[col], format='ISO8601', errors='coerce')
                      if dt_series.dt.tz is None:
                          record_df[col] = dt_series.dt.tz_localize('UTC', ambiguous='infer', nonexistent='NaT')
                      else:
                          record_df[col] = dt_series.dt.tz_convert('UTC')
          except Exception as e:
              logger.error(f"Error while converting {col} from {source}: {e}")

        if source == 'nextload':
            try:
                for col in ['post_date', 'last_updated']:
                    if col in record_df.columns:
                        dt_series = pd.to_datetime(record_df[col], format='ISO8601', errors='coerce')
                        if dt_series.dt.tz is None:
                            record_df[col] = dt_series.dt.tz_localize('UTC', ambiguous='infer', nonexistent='NaT')
                        else:
                            record_df[col] = dt_series.dt.tz_convert('UTC')
            except Exception as e:
                logger.error(f"Error while converting {col} from {source}: {e}")

            for col in ['pickup_date', 'drop_date']:
                if col in record_df.columns:
                    try:
                        dt_series = pd.to_datetime(record_df[col], errors='coerce')
                        if dt_series.dt.tz is None:
                            record_df[col] = dt_series.dt.tz_localize('UTC', ambiguous='infer', nonexistent='NaT')
                        else:
                            record_df[col] = dt_series.dt.tz_convert('UTC')
                    except Exception as e:
                        logger.error(f"Error while converting {col} from {source}: {e}")

            record_df['rate'] = record_df['rate'] / 100.0
            record_df['rate_per_mile'] = np.where(
                (record_df['rate'].isna()) | (record_df['estimated_distance'].isna()) | (record_df['estimated_distance'] <= 0),
                0.00,
                np.round(record_df['rate'] / record_df['estimated_distance'], 2)
            )
            record_df['DOT'] = pd.to_numeric(record_df['DOT'], errors='coerce').round().astype('Int64')
            record_df['MC'] = pd.to_numeric(record_df['MC'], errors='coerce').round().astype('Int64')


        record_df['drop_zone'] = record_df['drop_state'].map(zone_map)
        record_df['pickup_zone'] = record_df['pickup_state'].map(zone_map)
        record_df['city_lane'] = record_df[['pickup', 'drop']].astype(str).agg(' -> '.join, axis=1)
        record_df['state_lane'] = record_df[['pickup_state', 'drop_state']].astype(str).agg(' -> '.join, axis=1)
        record_df['zone_lane'] = record_df[['pickup_zone', 'drop_zone']].astype(str).agg(' -> '.join, axis=1)

        df = _broker_info(record_df, factoring_df, source)

        # Safely convert rate and estimated_distance to numeric before any calculations
        record_df['rate'] = pd.to_numeric(record_df['rate'], errors='coerce')

        # Handle other numeric columns with robust error handling
        numeric_cols = ['load_weight', 'load_height', 'load_length', 'load_width']
        for col in numeric_cols:
            if col in record_df.columns:
                try:
                    record_df[col] = pd.to_numeric(record_df[col], errors='coerce').fillna(0).round().astype('Int64')
                except Exception as e:
                    logger.error(f"Failed to convert '{col}' to numeric for source '{source}': {e} - Data: {record_df[col].tolist()}")
                    record_df[col] = pd.Series(dtype='Int64')

        # Final conversions with robust error handling
        try:
            record_df['rate'] = record_df['rate'].fillna(0).round().astype('Int64')
        except Exception as e:
            logger.error(f"Failed to convert 'rate' to Int64: {e} - Data: {record_df['rate'].tolist()}")
            record_df['rate'] = pd.Series(dtype='Int64')

        try:
            record_df['estimated_distance'] = pd.to_numeric(record_df['estimated_distance'], errors='coerce').fillna(0).round().astype(int)
        except Exception as e:
            logger.error(f"Failed to convert 'estimated_distance' to Int64: {e} - Data: {record_df['estimated_distance'].tolist()}")
            record_df['estimated_distance'] = pd.Series(dtype='Int64')

        # Round the rate_per_mile to 2 decimal places
        record_df['rate_per_mile'] = pd.to_numeric(record_df.get('rate_per_mile', 0), errors='coerce').fillna(0).astype(np.float64).round(2)
        record_df = _broker_info(record_df, factoring_df, source)
        logger.info("Data enrichment completed successfully.")
        return record_df

    except Exception as e:
        logger.error(f"Error during data enrichment: {e}")
        return pd.DataFrame()

## helper funct

In [12]:
def _load_factoring_data() -> pd.DataFrame:
    """Loads and merges factoring data from multiple sources."""
    try:
        nextload_df = pd.read_json(NEXTLOAD_FACTORS_FILE)
        nextload_df = pd.json_normalize(nextload_df['accountInfoList'])
        nextload_df = nextload_df[['mcNumber', 'debtorStatus']].dropna()
        nextload_df['mcNumber'] = pd.to_numeric(nextload_df['mcNumber'], errors='coerce').round().astype('Int64')
        nextload_df = nextload_df.rename(columns={'mcNumber': 'MC', 'debtorStatus': 'apex_credit_score'})

        trucksmarter_df = pd.read_json(TRUCKS_FACTORS_FILE)
        trucksmarter_df = trucksmarter_df[['brokerId','legalName', 'riskRating','mcNumber', 'dotNumber']].dropna()
        trucksmarter_df[['mcNumber', 'dotNumber']] = trucksmarter_df[['mcNumber', 'dotNumber']].round().astype('Int64')
        trucksmarter_df = trucksmarter_df.rename(columns={'mcNumber': 'MC', 'riskRating': 'trucksmarter_credit_score', 'dotNumber':'DOT'})

        factoring_df = pd.merge(trucksmarter_df, nextload_df, on='MC', how='outer')
        factoring_df[['apex_credit_score','trucksmarter_credit_score']] = factoring_df[['apex_credit_score','trucksmarter_credit_score']].fillna('NF').replace({'N/A':'NF'})
        factoring_df['legalName'] = factoring_df['legalName'].astype(str)
        return factoring_df
    except FileNotFoundError as e:
        logger.error(f"Factoring data file not found: {e}")
        return pd.DataFrame()

def _load_zone_map() -> pd.Series:
    """Loads the zone mapping from a CSV file."""
    try:
        zone_df = pd.read_csv(ZONE_FILE)
        return zone_df.set_index('Abbreviation')['Zone']
    except FileNotFoundError as e:
        logger.error(f"Zone map file not found: {e}")
        return pd.Series(dtype='object')

def load_secrets(filepath: str) -> Dict[str, Any]:
    """Loads API secrets from a JSON file."""
    try:
        with open(filepath, 'r') as f:
            secrets = json.load(f)
        logger.info("Secrets loaded successfully.")
        return secrets
    except FileNotFoundError:
        logger.critical(f"Secrets file not found at: {filepath}. Please create it.")
        raise
    except json.JSONDecodeError:
        logger.critical(f"Error decoding JSON from secrets file: {filepath}. Check file format.")
        raise


## extractor

In [13]:
# === Save helper ===
def save_json(data, source, suffix=""):
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{source}_{suffix}_{ts_str}.json" if suffix else f"{source}_{ts_str}.json"
    out_path = BRONZE_DIR / source / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)



# --- Async Data Extractors ---
async def extract_nextload(session: aiohttp.ClientSession, nextload_queue: asyncio.Queue, secrets: Dict[str, Any]):
    """Extracts raw data from Nextload."""
    logger.info("[Nextload Extractor] Starting Nextload data extraction loop.")
    while True:
        try:
            new_posted_loads = {"loads": []}
            page = 1
            total_pages = 1

            while page <= total_pages:
                json_data = {
                    'nameCustomized': False,
                    'criteria': [],
                    'uuid': '5ed232e2-41e4-45be-9baa-7508ea30622b',
                    'page': page,
                    'totalPages': total_pages,
                }
                async with session.post(
                    'https://my.nextload.com/rest/loads/search',
                    cookies=secrets['NL_COOKIES'],
                    headers=secrets['NL_HEADERS'],
                    json=json_data
                ) as resp:
                    resp.raise_for_status()
                    data = await resp.json()
                    total_pages = data.get('totalPages', total_pages)

                for load in data.get('loads', []):
                    load_hash = load.get('hash')
                    if load_hash and not _check_if_exists_in_db(DUCKDB_PATH, 'freight_data', 'nextload', None, load_hash):
                        await nextload_queue.put(('nextload', load))
                        new_posted_loads['loads'].append(load)
                        # await asyncio.sleep(20)
                page += 1
                total_pages = data.get('totalPages', total_pages)

            if new_posted_loads['loads']:
              new_posted_loads['requestTime'] = data.get('requestTime')
              save_json(new_posted_loads, "nextload", f"page{page}")
              print(f"[Nextload] — {len(new_posted_loads['loads'])} new loads found and saved")

            logger.info(f"[Nextload Extractor] Page {page}/{total_pages} — {len(data.get('loads', []))} loads scanned")
            await asyncio.sleep(1) # Be a good citizen

        except aiohttp.ClientError as e:
            logger.error(f"[Nextload Extractor] HTTP or Client Error: {e}")
        except json.JSONDecodeError:
            logger.error(f"[Nextload Extractor] JSON Decode Error: Could not parse response.")
        except Exception as e:
            logger.error(f"[Nextload Extractor] Unexpected Error: {e}")

        logger.info(f"[Nextload Extractor] Cycle finished. Waiting {INTERVAL_NL} seconds.")
        await asyncio.sleep(INTERVAL_NL)

async def extract_trucksmarter(session: aiohttp.ClientSession, trucksmarter_queue: asyncio.Queue, secrets: Dict[str, Any]):
    """Extracts raw data from Trucksmarter."""
    logger.info("[Trucksmarter Extractor] Starting Trucksmarter data extraction loop.")
    while True:
        today = date.today()
        new_posted_loads = {"loads": []}
        for state in STATES:
            try:

                json_data = {
                    'pickupTimeRange': {'startDay': {'year': today.year, 'month': today.month, 'day': today.day}, 'numDays': 5},
                    'options': {'sort': {'field': 'createdAt', 'order': 'desc'}, 'includeMissingPriceLoads': True},
                    'requirements': {'only': [], 'never': []},
                    'filters': {'trailerTypes': ['BoxTruck','CargoVan','Conestoga','Flatbed','Reefer','StepDeck','Van',], 'pickupSearchLocation': {'state': [state]},},
                }
                async with session.post(
                    'https://api.trucksmarter.com/loads/searchV2Ungrouped',
                    cookies=secrets['TS_COOKIES'],
                    headers=secrets['TS_HEADERS'],
                    json=json_data,
                    timeout=60
                ) as resp:
                    resp.raise_for_status()
                    data = await resp.json()

                for load in data.get('loads', []):
                    source_id = str(load.get('id'))
                    if source_id and not _check_if_exists_in_db(DUCKDB_PATH, 'freight_data', 'trucksmarter', source_id, None):
                        await trucksmarter_queue.put(('trucksmarter', load))
                        new_posted_loads['loads'].append(load)
                        # await asyncio.sleep(20)
                logger.info(f"[Trucksmarter Extractor] {state} — {len(data.get('loads', []))} loads scanned")
                await asyncio.sleep(0.5)

            except aiohttp.ClientError as e:
                logger.error(f"[Trucksmarter Extractor] HTTP or Client Error for {state}: {e}")
            except json.JSONDecodeError:
                logger.error(f"[Trucksmarter Extractor] JSON Decode Error for {state}.")
            except Exception as e:
                logger.error(f"[Trucksmarter Extractor] Unexpected Error in {state}: {e}")

        if new_posted_loads['loads']:
            new_posted_loads['requestTime'] = str(date.today())
            save_json(new_posted_loads, "trucksmarter", f"{date.today().isoformat()}")
            print(f"[Trucksmarter] — {len(new_posted_loads['loads'])} new loads saved")

        logger.info(f"[Trucksmarter Extractor] Cycle finished. Waiting {INTERVAL_TS} seconds.")
        await asyncio.sleep(INTERVAL_TS)

# --- Async Workers and Consumers ---
async def _extract_nextload_record(nextload_queue: asyncio.Queue, equipment_map: Dict[str, str], zone_map: pd.Series, factoring_df: pd.DataFrame, enriched_data_queue: asyncio.Queue) -> None:
    """Worker to process and enrich Nextload records."""
    while True:
        source, load = await nextload_queue.get()
        # print("\n[Nextload Queue Output]")
        # print(f"  Load Reference: {load.get('referenceNumber')}")
        # print(f"  City Lane: {load.get('pick', {}).get('location', {}).get('city')}-{load.get('drop', {}).get('location', {}).get('city')}")
        # print(f"  Dates: {load.get('originalPostingDate')}")
        # print(f"  Rates: {load.get('rate')}")
        # print(f"  Weight: {load.get('loadSize', {}).get('weight')}")

        # print(f"[Nextload Worker] Processing new load from Nextload queue.")
        equipment_display_names = [
            eq.get("displayName")
            for eq in load.get("equipmentTypes", [])
            if eq.get("displayName") is not None
        ]
        if not equipment_display_names:
            equipment_display_names = [None]
        for equipment_type in equipment_display_names:
            record = {
                "source": "nextload",
                "source_id": load.get("id"),
                "load_reference": load.get("referenceNumber"),
                "post_date": load.get("originalPostingDate"),
                "last_updated": load.get("postingDate"),
                "pickup_id": load.get("pick", {}).get("id"),
                "pickup_city": load.get("pick", {}).get("location", {}).get("city"),
                "pickup_state": load.get("pick", {}).get("location", {}).get("state"),
                "pickup_country": load.get("pick", {}).get("location", {}).get("country"),
                "pickup_latitude": load.get("pick", {}).get("location", {}).get("latitude"),
                "pickup_longitude": load.get("pick", {}).get("location", {}).get("longitude"),
                "pickup_date": load.get("pick", {}).get("startDate"),
                "drop_id": load.get("drop", {}).get("id"),
                "drop_city": load.get("drop", {}).get("location", {}).get("city"),
                "drop_state": load.get("drop", {}).get("location", {}).get("state"),
                "drop_country": load.get("drop", {}).get("location", {}).get("country"),
                "drop_latitude": load.get("drop", {}).get("location", {}).get("latitude"),
                "drop_longitude": load.get("drop", {}).get("location", {}).get("longitude"),
                "drop_date": load.get("drop", {}).get("startDate"),
                "equipment_type": equipment_type,
                "load_length": load.get("loadSize", {}).get("length"),
                "load_weight": load.get("loadSize", {}).get("weight"),
                "load_height": load.get("loadSize", {}).get("height"),
                "load_width": load.get("loadSize", {}).get("width"),
                "rate": load.get("rate"),
                "comment": load.get("comment"),
                "contact_name": load.get("contactInfo", {}).get("dispatcherName"),
                "contact_phone": load.get("contactInfo", {}).get("phoneNumber"),
                "contact_email": load.get("user", {}).get("userName"),
                "contact_fax": load.get("user", {}).get("fax"),
                "company_name": load.get("user", {}).get("companyName"),
                "company_email": load.get("user", {}).get("email"),
                "MC": next((a.get("numericValue") for a in load.get("user", {}).get("authorities", []) if (a.get("type") or {}).get("name") == "MC"), None),
                "DOT": next((a.get("numericValue") for a in load.get("user", {}).get("authorities", []) if (a.get("type") or {}).get("name") == "DOT"), None),
                "estimated_distance": load.get("estimatedDistance"),
                "hash": load.get("hash"),
            }
            enriched_record = enrich_record(record, source, factoring_df, equipment_map, zone_map)
            logger.info("[Nextload Worker] Enriched completed.")

            if not enriched_record.empty:
                await enriched_data_queue.put(enriched_record)
                logger.info("[Nextload Worker] Enriched record added to the final queue.")
        nextload_queue.task_done()

async def _extract_trucksmarter_record(trucksmarter_queue: asyncio.Queue, equipment_map: Dict[str, str], zone_map: pd.Series, factoring_df: pd.DataFrame, enriched_data_queue: asyncio.Queue) -> None:
    """Worker to process and enrich Trucksmarter records."""
    while True:
        source, load = await trucksmarter_queue.get()
        pickup_stop = next((s for s in load.get('stops', []) if s.get('type') == 'Pickup'), None)
        delivery_stop = next((s for s in load.get('stops', []) if s.get('type') == 'Delivery'), None)
        pickup_address = pickup_stop.get("address", {}) if pickup_stop else {}
        delivery_address = delivery_stop.get("address", {}) if delivery_stop else {}

        # print("\n[Trucksmarter Queue Output]")
        # print(f"  Load Reference: {load.get('brokerLoadId')}")
        # print(f"  City Lane: {pickup_address.get('city')}-{delivery_address.get('city')}")
        # print(f"  Dates: {load.get('createdAt')}")
        # print(f"  Rates: {load.get('price')}")
        # print(f"  Weight: {load.get('weight')}")

        # print(f"[Trucksmarter Worker] Processing new load from Trucksmarter queue.")
        equipment_info = load.get("equipment", {}) or {}
        equipment_types = equipment_info.get("trailerTypes", [])
        if not equipment_types:
            equipment_types = [None]
        for equipment_type in equipment_types:
            record = {
                "source": 'trucksmarter',
                "source_id": load.get("id"),
                "load_reference": load.get("brokerLoadId"),
                "post_date": load.get("createdAt"),
                "last_updated": load.get("lastExtractedAt"),
                "pickup_id": pickup_stop.get("stopIndex") if pickup_stop else None,
                "pickup_city": pickup_address.get("city").title() if pickup_address.get("city") else None,
                "pickup_state": pickup_address.get("state"),
                "pickup_country": pickup_address.get("countryIso2"),
                "pickup_latitude": pickup_stop.get("latitude") if pickup_stop else None,
                "pickup_longitude": pickup_stop.get("longitude") if pickup_stop else None,
                "pickup_date": (load.get("pickup") or {}).get("appointmentStartTime") if pickup_stop else None,
                "display_timezone": (load.get("displayTimezone") or {}).get("appointmentStartTime") if pickup_stop else None,
                "drop_id": delivery_stop.get("stopIndex") if delivery_stop else None,
                "drop_city": delivery_address.get("city").title() if delivery_address.get("city") else None,
                "drop_state": delivery_address.get("state"),
                "drop_country": delivery_address.get("countryIso2"),
                "drop_latitude": delivery_stop.get("latitude") if delivery_stop else None,
                "drop_longitude": delivery_stop.get("longitude") if delivery_stop else None,
                "drop_date": (load.get("delivery") or {}).get("appointmentStartTime") if delivery_stop else None,
                "equipment_type": equipment_type,
                # "is_full_load": None,
                "load_length": equipment_info.get("length"),
                "load_weight": load.get("weight"),
                "load_height": equipment_info.get("height"),
                "load_width": equipment_info.get("width"),
                "rate": load.get("price"),
                "rate_per_mile": load.get("ratePerMile"),
                "comment": f"{(load.get('pickup') or {}).get('note', '')}, {(load.get('delivery') or {}).get('note', '')}",
                "contact_name": None,
                "contact_phone": load.get("bookingPhoneNumber"),
                "contact_email": load.get("biddingEmail"),
                "contact_fax": None,
                "company_name": load.get("broker"),
                "company_email": load.get("biddingEmail"),
                "estimated_distance": load.get("distance"),
                "hash": None,
            }
            enriched_record = enrich_record(record, source, factoring_df, equipment_map, zone_map, country_fix=True)
            if not enriched_record.empty:
                await enriched_data_queue.put(enriched_record)
                logger.info("[Trucksmarter Worker] Enriched record added to the final queue.")
        trucksmarter_queue.task_done()


In [ ]:
# -*- coding: utf-8 -*-
from datetime import datetime, timezone
async def data_consumer(enriched_data_queue: asyncio.Queue) -> None:
    """Consumes enriched data, inserts it into DuckDB, and runs an analytical query."""
    logger.info("[Final Consumer] Ready to process enriched data.")
    # print("[Final Consumer] Ready to process enriched data.")
    table_name = 'freight_data'

    # --- Option 1: Batching by Record Count (Uncomment to use) ---
    # batch_size = 100
    # batch = []

    # --- Option 2: Batching by Time (Commented out) ---
    # batch = []
    # last_write_time = asyncio.get_event_loop().time()
    # batch_interval_seconds = 60
    column_order = [
    'source', 'source_id', 'load_reference', 'post_date', 'last_updated',
    'pickup_id', 'pickup_city', 'pickup_state', 'pickup_country', 'pickup_latitude',
    'pickup_longitude', 'pickup_date', 'drop_id', 'drop_city', 'drop_state',
    'drop_country', 'drop_latitude', 'drop_longitude', 'drop_date',
    'equipment_type',  'load_length', 'load_weight',
    'load_height', 'load_width', 'rate', 'comment', 'contact_name',
    'contact_phone', 'contact_email', 'contact_fax', 'company_name',
    'company_email', 'MC', 'DOT', 'estimated_distance', 'hash', 'pickup',
    'drop',  'drop_zone', 'pickup_zone', 'rate_per_mile', 'apex_credit_score',
    'trucksmarter_credit_score', 'city_lane', 'state_lane', 'zone_lane'
]
    while True:
        record_df = await enriched_data_queue.get()
        if record_df.empty:
            # enriched_data_queue.task_done()
            continue
        try:
            # print(f'col diff set(column_order) - set(record_df.columns.tolist(): {set(column_order) - set(record_df.columns.tolist())}')
            # print(f'col diff set(record_df.columns.tolist() - set(column_order)): {set(record_df.columns.tolist()) - set(column_order)}')
            record_df = record_df.reindex(columns=column_order, fill_value=None)

            # --- Single Record Insert (Current Implementation) ---
            with duckdb.connect(database=DUCKDB_PATH, read_only=False) as conn:
                conn.append(table_name, record_df)
                logger.info(f"[Final Consumer] Record successfully inserted into DuckDB table '{table_name}'")
                # print(f"[Final Consumer] Record successfully inserted into DuckDB table '{table_name}'")

            # Check for active/changing loads on the newly processed data
            columns_for_dectecting_active_loads = ['city_lane','equipment_type','pickup_date']
            active_loads_df = track_active_or_changing_loads_sql(DUCKDB_PATH, record_df, columns_for_dectecting_active_loads)
            if not active_loads_df.empty:
                for _, result in active_loads_df.iterrows():
                  if int(result['posted_count']) > 1:
                    days = result['days_since_post']
                    if days == 0:
                        posted_str = "today"
                    else:
                        plural = "s" if days != 1 else ""
                        posted_str = f"{days} day{plural} ago"


                    logger.info(
                        f"load Id: {result['load_reference']}\n"
                        f"Lane: {result['city_lane']}\n"
                        f"Zone Lane: {result['zone_lane']}\n"
                        f"State Lane: {result['state_lane']}\n"
                        f"Broker: {result['company_name']}\n"
                        f"MC#: {result['MC']}\n"
                        f"Posted: {result['posted_count']} time(s)\n"
                        f"First Posted: {result['days_since_post']} day(s) ago\n"
                        f"Weight: {result['load_weight']}\n"
                        f"Equipment: {result['equipment_type']}\n"
                        f"Pickup Date: {result['pickup_date']}\n"
                        f"Rate: {result['first_rate']} → {result['last_rate']}"
                    )
                    data_to_log = (
                        f" ------------------{datetime.now(timezone.utc).date()}-------------------\n"
                        f"load Id: {result['load_reference']}\n"
                        f"Lane: {result['city_lane']}\n"
                        f"Zone Lane: {result['zone_lane']}\n"
                        f"State Lane: {result['state_lane']}\n"
                        f"Broker: {result['company_name']}\n"
                        f"Apex Credit Rating: {result['apex_credit_score']}\n"
                        f"Trucksmarter Credit Rating: {result['trucksmarter_credit_score']}\n"
                        f"MC#: {result['MC']}\n"
                        f"Posted: {result['posted_count']} time(s)\n"
                        f"First Posted: {posted_str}\n"
                        f"Weight: {result['load_weight']}\n"
                        f"Equipment: {result['equipment_type']}\n"
                        f"Pickup Date: {str(result['pickup_date']).split(' ')[0]}\n"
                        f"Rate: {result['first_rate']} → {result['last_rate']}\n"
                        f"Comment: {result['comment']}\n"
                        f" ----------------------------------------\n\n\n"
                    )
                    print(data_to_log)
                    with open('/content/drive/MyDrive/freight_analysis/track_Active_changing_loads.txt', 'a') as f:
                        f.write(data_to_log)
                    # print(data_to_log
            else:
                logger.info(f"No active/changing loads found for the new records.")
                # print(f"No active/changing loads found for the new records.")

        except Exception as e:
            logger.error(f"[Final Consumer] Unexpected Error during data write or query: {e}")
            print(f"[Final Consumer] Unexpected Error during data write or query: {e}")

        enriched_data_queue.task_done()

# --- Main Orchestrator ---
async def main():
    """Main function to run the data pipeline."""
    logger.info("[Main] Starting the freight data pipeline.")

    secrets = load_secrets(SECRETS_FILE)
    _load_duckdb_table(DUCKDB_PATH, 'freight_data')
    factoring_df = _load_factoring_data()
    zone_map = _load_zone_map()

    nextload_queue = asyncio.Queue()
    trucksmarter_queue = asyncio.Queue()
    enriched_data_queue = asyncio.Queue()

    tasks = [] # Define the list outside the try block

    try:
        async with aiohttp.ClientSession() as session:
            logger.info("[Main] Launching all tasks.")
            tasks = [
                # Raw data extractors
                asyncio.create_task(extract_nextload(session, nextload_queue, secrets)),
                asyncio.create_task(extract_trucksmarter(session, trucksmarter_queue, secrets)),
                # Enrichment workers
                asyncio.create_task(_extract_nextload_record(nextload_queue, STANDARDIZE_EQUIPMENT_MAP, zone_map, factoring_df, enriched_data_queue)),
                asyncio.create_task(_extract_trucksmarter_record(trucksmarter_queue, STANDARDIZE_EQUIPMENT_MAP, zone_map, factoring_df, enriched_data_queue)),
                # Final consumer
                asyncio.create_task(data_consumer(enriched_data_queue))
            ]
            await asyncio.gather(*tasks)
    except KeyboardInterrupt:
        print("\nKeyboard interrupt received. Cancelling tasks...")
        for task in tasks:
            task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True) # Await cancellation
        print("All tasks are now cancelled.")

if __name__ == "__main__":
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except KeyboardInterrupt:
        logger.info("Stopped by user.")
    except Exception as e:
        logger.critical(f"A critical error occurred: {e}")

 ------------------2025-08-28-------------------
load Id: 121406654
Lane: Mangonia Park,FL,USA -> East Bend,NC,USA
Zone Lane: Z3 -> Z2
State Lane: FL -> NC
Broker: KNW Holdings INC
Apex Credit Rating: A
Trucksmarter Credit Rating: B
MC#: 660086
Posted: 2 time(s)
First Posted: today
Weight: 11602
Equipment: Flatbed
Pickup Date: 2025-08-28
Rate: 951 → 951
Comment: None, None
 ----------------------------------------



 ------------------2025-08-28-------------------
load Id: 121406654
Lane: Mangonia Park,FL,USA -> East Bend,NC,USA
Zone Lane: Z3 -> Z2
State Lane: FL -> NC
Broker: KNW Holdings INC
Apex Credit Rating: A
Trucksmarter Credit Rating: B
MC#: 660086
Posted: 2 time(s)
First Posted: today
Weight: 11602
Equipment: HotShot
Pickup Date: 2025-08-28
Rate: 951 → 951
Comment: None, None
 ----------------------------------------



 ------------------2025-08-28-------------------
load Id: 57b99dd3735b9a42a1ba3256a29bb418d79998a9
Lane: Columbia City,OR,USA -> Gunnison,CO,USA
Zone Lane: Z9

ERROR:__main__:[Nextload Extractor] HTTP or Client Error: [Errno 32] Broken pipe


 ------------------2025-08-28-------------------
load Id: b57d6a58e8b23ee078cfffca13b26adc27919eca
Lane: Long Island,VA,USA -> Clarksville,TN,USA
Zone Lane: Z2 -> Z3
State Lane: VA -> TN
Broker: Bieri Brokerage CO INC
Apex Credit Rating: A
Trucksmarter Credit Rating: A
MC#: 333769
Posted: 2 time(s)
First Posted: today
Weight: 48000
Equipment: Flatbed
Pickup Date: 2025-08-28
Rate: 1350 → 1350
Comment: MUST DROP BY SAT, None
 ----------------------------------------



 ------------------2025-08-28-------------------
load Id: b57d6a58e8b23ee078cfffca13b26adc27919eca
Lane: Long Island,VA,USA -> Clarksville,TN,USA
Zone Lane: Z2 -> Z3
State Lane: VA -> TN
Broker: Bieri Brokerage CO INC
Apex Credit Rating: A
Trucksmarter Credit Rating: A
MC#: 333769
Posted: 2 time(s)
First Posted: today
Weight: 48000
Equipment: Step Deck
Pickup Date: 2025-08-28
Rate: 1350 → 1350
Comment: MUST DROP BY SAT, None
 ----------------------------------------



 ------------------2025-08-28-------------------
load 

## Main

In [ ]:
# async def graceful_shutdown(loop, signal=None):
#     """
#     Cancel all tasks and stop the event loop.
#     """
#     logger.info("Received shutdown signal. Initiating graceful shutdown...")
#     tasks = [t for t in asyncio.all_tasks(loop) if t is not asyncio.current_task()]
#     for task in tasks:
#         task.cancel()

#     logger.info(f"Cancelling {len(tasks)} tasks...")
#     await asyncio.gather(*tasks, return_exceptions=True)
#     loop.stop()

# # --- Main Orchestrator ---
# async def main():
#     """Main function to run the data pipeline."""
#     logger.info("[Main] Starting the freight data pipeline.")

#     secrets = load_secrets(SECRETS_FILE)
#     _load_duckdb_table(DUCKDB_PATH, 'freight_data')
#     factoring_df = _load_factoring_data()
#     zone_map = _load_zone_map()

#     nextload_queue = asyncio.Queue()
#     trucksmarter_queue = asyncio.Queue()
#     enriched_data_queue = asyncio.Queue()

#     async with aiohttp.ClientSession() as session:
#         logger.info("[Main] Launching all tasks.")
#         tasks = [
#             # Raw data extractors
#             asyncio.create_task(extract_nextload(session, nextload_queue, secrets)),
#             asyncio.create_task(extract_trucksmarter(session, trucksmarter_queue, secrets)),
#             # Enrichment workers
#             asyncio.create_task(_extract_nextload_record(nextload_queue, STANDARDIZE_EQUIPMENT_MAP, zone_map, factoring_df, enriched_data_queue)),
#             asyncio.create_task(_extract_trucksmarter_record(trucksmarter_queue, STANDARDIZE_EQUIPMENT_MAP, zone_map, factoring_df, enriched_data_queue)),
#             # Final consumer
#             asyncio.create_task(data_consumer(enriched_data_queue))
#         ]

#         await asyncio.gather(*tasks)

# if __name__ == "__main__":
#     loop = asyncio.get_event_loop()
#     try:
#         # Use loop.run_until_complete() for direct event loop management
#         loop.run_until_complete(main())
#     except KeyboardInterrupt:
#         logger.info("Program interrupted by user. Shutting down gracefully...")
#         # Get all tasks and cancel them
#         tasks = [t for t in asyncio.all_tasks(loop) if t is not asyncio.current_task()]
#         for task in tasks:
#             task.cancel()

#         # Run until all tasks are done or have raised an exception
#         loop.run_until_complete(asyncio.gather(*tasks, return_exceptions=True))
#     finally:
#         # loop.close()
#         logger.info("Event loop closed. Program terminated.")




# New Section

## end